# Multi-Agent Clinical Triage System — Week 8
### Stella Oiro — Andela AI Engineering Bootcamp

---

**The progression:**
- Week 5: Built a RAG chatbot over clinical guidelines (ChromaDB + GPT-4.1-mini)
- Week 7: Fine-tuned Llama 3.2 3B with QLoRA for offline triage classification
- Week 8: Deploy the fine-tuned model on Modal, combine with RAG + GPT into a multi-agent ensemble, add autonomous planning

**The result:** An autonomous triage assistant that classifies patient urgency using three independent sources of intelligence, escalates Immediate cases via push notification, and runs the offline model without sending patient data to any cloud API.

---

## Architecture

```
Clinical Presentation
        │
        ▼
  AutonomousAgent  (GPT-4.1-mini with tool calling)
        │
        ▼
  EnsembleAgent   (weighted vote)
   ┌────┼────────────┐
   ▼    ▼            ▼
Specialist  RAG   Frontier
(Modal/    (Chroma  (GPT
 Llama)   + GPT)   zero-shot)
        │
        ▼
  Triage Decision
        │
    (if Immediate)
        ▼
  MessengerAgent → Push notification to on-call doctor
```

| Agent | Accuracy | Notes |
|---|---|---|
| SpecialistAgent (fine-tuned Llama) | ~89% | Runs offline, no data leaves hospital |
| RAGAgent (guidelines + GPT) | ~85% | Evidence-grounded |
| FrontierAgent (GPT-4.1-mini) | ~72% | Fast baseline |
| EnsembleAgent (weighted vote) | ~91% | Best overall |

---

## Order of Play

**Part 1** — Setup + build clinical vector store  
**Part 2** — Deploy fine-tuned model on Modal (SpecialistAgent)  
**Part 3** — Test each agent individually  
**Part 4** — EnsembleAgent: weighted vote + safety escalation  
**Part 5** — AutonomousAgent: tool-calling orchestrator + Gradio UI  

---
## Setup

In [ ]:
# Install dependencies
%pip install -q openai chromadb sentence-transformers langchain langchain-community gradio requests python-dotenv

In [ ]:
import os
import sys
from dotenv import load_dotenv

load_dotenv(override=True)

# Add this folder to path so we can import agents/
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

# Configuration
HF_MODEL_NAME = "AcharO/clinical-triage-llama"
VECTOR_DB_PATH = "clinical_vector_db"

assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY not set'
print('Environment ready.')

---
# Part 1 — Build Clinical Vector Store

The RAGAgent needs a ChromaDB store of clinical guidelines. If you completed Week 5, you can reuse that store. Otherwise, we build a minimal one here from embedded clinical knowledge.

The guidelines cover: hypertension, T2DM, asthma, heart failure, sepsis, triage protocols.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# Core clinical triage guidelines — embedded directly so the notebook is self-contained
GUIDELINES = [
    # Immediate
    "IMMEDIATE triage (see now): Cardiac arrest, ventricular fibrillation, pulseless electrical activity. Start CPR immediately. Defibrillate if shockable rhythm.",
    "IMMEDIATE triage (see now): Severe anaphylaxis — angioedema, stridor, hypotension after allergen exposure. Give IM adrenaline 0.5mg immediately.",
    "IMMEDIATE triage (see now): Acute MI with haemodynamic instability — crushing chest pain, diaphoresis, BP <90 systolic, ST elevation. Activate cath lab.",
    "IMMEDIATE triage (see now): Unresponsive patient, GCS ≤8, not protecting airway. Prepare for intubation.",
    "IMMEDIATE triage (see now): Tension pneumothorax — absent breath sounds, tracheal deviation, hypotension, distended neck veins. Needle decompression urgently.",
    "IMMEDIATE triage (see now): Status epilepticus — continuous seizure >5 minutes or recurrent without recovery. IV benzodiazepines.",
    "IMMEDIATE triage (see now): Massive haemorrhage — penetrating trauma, uncontrolled external bleeding, haemodynamic shock. Apply tourniquet/pressure.",
    # Urgent
    "URGENT triage (within 30 minutes): High fever in infant <3 months (temp >38°C). Risk of sepsis. Full septic screen and empirical antibiotics.",
    "URGENT triage (within 30 minutes): Moderate chest pain without haemodynamic compromise. 12-lead ECG within 10 minutes. Troponin at 0 and 3 hours.",
    "URGENT triage (within 30 minutes): Acute severe asthma — unable to complete sentences, O2 sat <92%, PEFR <50% predicted. Nebulised salbutamol + oxygen.",
    "URGENT triage (within 30 minutes): Acute confusion/delirium in elderly patient. Assess for infection, metabolic cause, medication effect.",
    "URGENT triage (within 30 minutes): Suspected stroke — FAST positive (facial droop, arm weakness, speech difficulty, time). CT within 25 minutes of arrival.",
    "URGENT triage (within 30 minutes): Severe abdominal pain with peritoneal signs. NPO, IV access, surgical review.",
    "URGENT triage (within 30 minutes): Moderate dyspnoea, O2 sat 90-94%. Supplemental oxygen, identify cause.",
    # Semi-urgent
    "SEMI-URGENT triage (within 1 hour): Mild to moderate asthma, O2 sat >95%, able to talk in sentences. Salbutamol inhaler, reassess in 20 minutes.",
    "SEMI-URGENT triage (within 1 hour): Laceration requiring sutures, bleeding controlled, no tendon/nerve injury suspected.",
    "SEMI-URGENT triage (within 1 hour): Urinary tract infection with moderate symptoms. MSU, oral antibiotics if uncomplicated.",
    "SEMI-URGENT triage (within 1 hour): Suspected fracture with intact neurovascular status. X-ray, analgesia.",
    "SEMI-URGENT triage (within 1 hour): Moderate pain (NRS 5-7), vital signs stable. Analgesia, assess cause.",
    # Non-urgent
    "NON-URGENT triage (can wait): Minor upper respiratory tract infection in otherwise well adult. Symptomatic management, no antibiotics.",
    "NON-URGENT triage (can wait): Routine medication query or prescription request. Redirect to GP if possible.",
    "NON-URGENT triage (can wait): Minor skin irritation or rash, stable vital signs, no systemic symptoms.",
    "NON-URGENT triage (can wait): Mild musculoskeletal pain, no trauma, neurovascularly intact. RICE, analgesia.",
]

print(f"Building vector store with {len(GUIDELINES)} clinical guideline chunks...")

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = embedder.encode(GUIDELINES).tolist()

chroma = chromadb.PersistentClient(path=VECTOR_DB_PATH)

# Create fresh collection
try:
    chroma.delete_collection("clinical_guidelines")
except:
    pass
collection = chroma.create_collection("clinical_guidelines")
collection.add(
    documents=GUIDELINES,
    embeddings=embeddings,
    ids=[f"guideline_{i}" for i in range(len(GUIDELINES))],
)

print(f"Vector store ready: {collection.count()} chunks in {VECTOR_DB_PATH}/")

In [ ]:
# Test retrieval
query = "72yo male, sudden crushing chest pain, sweating, BP 80/50"
q_embedding = embedder.encode(query).tolist()
results = collection.query(query_embeddings=[q_embedding], n_results=3)

print(f"Query: {query}\n")
print("Top 3 retrieved guidelines:")
for i, doc in enumerate(results['documents'][0]):
    print(f"  {i+1}. {doc[:100]}...")

---
# Part 2 — Deploy Fine-Tuned Model on Modal (SpecialistAgent)

We deploy the Llama 3.2 3B model fine-tuned in Week 7 as a serverless GPU endpoint on Modal.

**Before running this section:**
1. Set up Modal: `uv run modal token set --token-id ak-... --token-secret as-...`
2. Add a Modal secret named `huggingface-secret` with key `HF_TOKEN`
3. Update `HF_MODEL_NAME` above to your Week 7 model repo

In [ ]:
# Update HF_MODEL_NAME in modal_triage_service.py, then deploy:
import subprocess
result = subprocess.run(
    ["uv", "run", "modal", "deploy", "modal_triage_service.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

In [ ]:
# Test the Modal service directly
import modal

TriageClassifier = modal.Cls.from_name("clinical-triage-service", "TriageClassifier")
classifier = TriageClassifier()

test_case = "72yo male, sudden crushing chest pain, sweating profusely, BP 85/50"
result = classifier.classify.remote(test_case)
print(f"Presentation: {test_case}")
print(f"SpecialistAgent: {result}  (expected: Immediate)")

---
# Part 3 — Test Each Agent Individually

In [ ]:
from agents.specialist_agent import SpecialistAgent
from agents.rag_agent import RAGAgent
from agents.frontier_agent import FrontierAgent

specialist = SpecialistAgent()
rag        = RAGAgent(VECTOR_DB_PATH)
frontier   = FrontierAgent()

In [ ]:
TEST_CASES = [
    ("72yo male, crushing chest pain, sweating, BP 85/50, pallor", "Immediate"),
    ("4yo female, fever 39.8°C for 2 days, irritable, not eating", "Urgent"),
    ("28yo male, laceration to right forearm, bleeding controlled", "Semi-urgent"),
    ("55yo female, mild headache for 3 days, no fever, no neuro symptoms", "Non-urgent"),
    ("Infant 8 months, found unresponsive by parent, not breathing", "Immediate"),
]

print(f"{'Presentation':<55} {'Expected':<12} {'Specialist':<12} {'RAG':<12} {'Frontier':<12}")
print("-" * 103)

for presentation, expected in TEST_CASES:
    s = specialist.classify(presentation)
    r = rag.classify(presentation)
    f = frontier.classify(presentation)
    print(f"{presentation[:53]:<55} {expected:<12} {s:<12} {r:<12} {f:<12}")

---
# Part 4 — EnsembleAgent: Weighted Vote + Safety Escalation

Weights are proportional to each agent's observed accuracy:
- SpecialistAgent: 0.45 (89% accuracy, offline)
- RAGAgent: 0.35 (85% accuracy, evidence-grounded)
- FrontierAgent: 0.20 (72% accuracy, zero-shot)

**Safety rule:** If ANY agent votes Immediate, the ensemble returns Immediate.
Under-triage of a life-threatening case is a worse error than over-triage.

In [ ]:
from agents.ensemble_agent import EnsembleAgent

ensemble = EnsembleAgent(VECTOR_DB_PATH)

In [ ]:
print("Ensemble results:\n")
for presentation, expected in TEST_CASES:
    decision, votes = ensemble.classify(presentation)
    match = "✓" if decision == expected else "✗"
    print(f"{match} [{expected}] → [{decision}]")
    print(f"  Votes: {votes}")
    print(f"  Presentation: {presentation[:70]}")
    print()

In [ ]:
# Accuracy on test cases
correct = sum(
    1 for p, expected in TEST_CASES
    if ensemble.classify(p)[0] == expected
)
print(f"Ensemble accuracy on test cases: {correct}/{len(TEST_CASES)} = {correct/len(TEST_CASES):.0%}")

---
# Part 5 — AutonomousAgent + Gradio UI

The AutonomousAgent uses GPT-4.1-mini with tool calling to orchestrate the pipeline:
1. LLM decides to call `triage_patient` → EnsembleAgent classifies
2. If Immediate → LLM calls `notify_doctor` → MessengerAgent sends push notification
3. LLM returns structured reasoning

Set `PUSHOVER_USER` and `PUSHOVER_TOKEN` in `.env` to enable real notifications.

In [ ]:
from agents.autonomous_agent import AutonomousAgent

agent = AutonomousAgent(VECTOR_DB_PATH)

In [ ]:
# Test the autonomous agent
result = agent.run("72yo male, crushing chest pain, sweating, BP 85/50")

print(f"Triage Level: {result['triage_level']}")
print(f"Agent Votes:  {result['votes']}")
print(f"Reasoning:    {result['reasoning']}")

In [ ]:
import gradio as gr

LEVEL_STYLES = {
    "Immediate":   ("#d32f2f", "IMMEDIATE — Life-threatening. See patient NOW."),
    "Urgent":      ("#f57c00", "URGENT — See within 30 minutes."),
    "Semi-urgent": ("#f9a825", "SEMI-URGENT — See within 1 hour."),
    "Non-urgent":  ("#388e3c", "NON-URGENT — Can wait. Manage in order of arrival."),
    "Unknown":     ("#9e9e9e", "Unable to classify. Please review manually."),
}

def triage(presentation: str) -> tuple[str, str]:
    if not presentation.strip():
        return "Please enter a clinical presentation.", ""

    result = agent.run(presentation.strip())
    level = result["triage_level"]
    votes = result["votes"]
    reasoning = result.get("reasoning", "")

    color, label = LEVEL_STYLES.get(level, LEVEL_STYLES["Unknown"])

    decision_html = f"""
    <div style="background:{color};color:white;padding:20px;border-radius:10px;font-size:20px;font-weight:bold;margin-bottom:12px">
        {label}
    </div>
    <div style="background:#f5f5f5;padding:14px;border-radius:8px;font-family:monospace;font-size:13px">
        <b>Agent votes:</b><br>
        {'<br>'.join(f'• {k}: <b>{v}</b>' for k, v in votes.items())}
    </div>
    """
    return decision_html, reasoning


EXAMPLES = [
    ["72yo male, sudden crushing chest pain, sweating, BP 85/50, pallor"],
    ["4yo female, fever 39.8°C for 2 days, irritable, not eating or drinking"],
    ["28yo male, laceration to right forearm from knife, bleeding now controlled"],
    ["55yo female, mild headache for 3 days, no fever, no neurological symptoms"],
    ["Infant 8 months, found unresponsive by parent, apnoeic"],
    ["35yo male, moderate shortness of breath, O2 sat 93%, audible wheeze"],
]

with gr.Blocks(theme=gr.themes.Soft(), title="Clinical Triage System") as demo:
    gr.Markdown("""
    # Multi-Agent Clinical Triage System
    **Fine-tuned Llama (offline) + RAG over clinical guidelines + GPT-4.1-mini — orchestrated by an autonomous planning agent**

    Enter a brief clinical presentation as a triage nurse would record it.
    """)

    with gr.Row():
        inp = gr.Textbox(
            label="Clinical Presentation",
            placeholder="e.g. 65yo female, sudden onset worst headache of her life, vomiting, neck stiffness...",
            lines=3,
        )
    btn = gr.Button("Triage", variant="primary", size="lg")

    with gr.Row():
        with gr.Column(scale=2):
            decision_out = gr.HTML(label="Triage Decision")
        with gr.Column(scale=1):
            reasoning_out = gr.Textbox(label="Agent Reasoning", lines=6)

    gr.Examples(examples=EXAMPLES, inputs=inp)
    btn.click(triage, inputs=inp, outputs=[decision_out, reasoning_out])

demo.launch(share=True)